<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/1_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Bit LLM

In [52]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [53]:
class BitLinear(nn.Linear):
  def forward(self, x: torch.Tensor):
    # Scale input tensor down to 8 bits
    i_scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_quant = torch.round(x * 127 / i_scale).clamp(-128, 127)

    # Scale layer weights to ternary -,+,0
    w = self.weight
    w_scale = w.abs().mean()
    w_quant = torch.round(w / (w_scale + 1e-5)).clamp(-1, 1)
    w_ternary = w + (w_quant - w).detach()

    out_t = F.linear(x_quant, w_ternary, self.bias)

    # Scale output tensor up to 8bits
    out_t = out_t * (i_scale / 127) * (w_scale)
    return out_t

Tests for the BitLinear class

In [54]:
bit_linear_layer = BitLinear(10, 5)
x = torch.randn(1, 10)
output = bit_linear_layer(x)

assert isinstance(output, torch.Tensor), "Assertion Failed: Output should be a torch.Tensor"
assert output.shape == (1, 5), f"Assertion Failed: Output shape mismatch. Expected (1, {5}), got {output.shape}"
assert not torch.isnan(output).any(), "Assertion Failed: Output contains NaN values"
assert not torch.isinf(output).any(), "Assertion Failed: Output contains Inf values"
print(f"Output tensor (first 5 elements): {output.flatten()[:5].tolist()}")

Output tensor (first 5 elements): [0.6085319519042969, -0.4019235074520111, 0.18807226419448853, -0.033560581505298615, -0.07161437720060349]


Replace Linear Layers with BitLinear

In [55]:
def replace_linears_with_bitlinear(model: nn.Module):
  for name, module in model.named_children():
    if isinstance(module, nn.Linear):
      has_bias = module.bias is not None
      bit_layer = BitLinear(module.in_features, module.out_features, bias=has_bias)
      setattr(model, name, bit_layer)
    else:
      replace_linears_with_bitlinear(module)

Test the replacement function

In [56]:
class TestModule(nn.Module):
  def __init__(self):
    super().__init__()
    self.l1, self.relu, self.l2 = nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5)

  def forward(self, x):
    return self.l2(self.relu(self.l1(x)))

original_model = TestModule()

print("Original model structure:")
print(original_model)

replace_linears_with_bitlinear(original_model)

print("\nModel structure after replacement:")
print(original_model)

assert isinstance(original_model.l1, BitLinear), "l1 was not replaced by BitLinear"
assert isinstance(original_model.l2, BitLinear), "l2 was not replaced by BitLinear"

Original model structure:
TestModule(
  (l1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): Linear(in_features=20, out_features=5, bias=True)
)

Model structure after replacement:
TestModule(
  (l1): BitLinear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): BitLinear(in_features=20, out_features=5, bias=True)
)


Setitng up the Transformer Blocks